# Best-of-N

**Paper**: [WebGPT: Browser-assisted question-answering with human feedback](https://arxiv.org/abs/2112.09332)

**Authors**: Reiichiro Nakano, Jacob Hilton, Suchir Balaji, Jeff Wu, Long Ouyang, Christina Kim, Christopher Hesse, Shantanu Jain, Vineet Kosaraju, William Saunders, Xu Jiang, Karl Cobbe, Tyna Eloundou, Gretchen Krueger, Kevin Button, Matthew Knight, Benjamin Chess, John Schulman

Best-of-N sampling is a standard inference-time alignment baseline. It samples several full-length continuations from the base model and returns the single highest-scoring one under a supplied sequence scorer. Pairing the scorer with a majority-vote scorer recovers self-consistency whereas pairing it with a metric scorer gives metric-guided reranking.

Best-of-N is a decoding driver in our toolkit built on the generic search driver, mapping onto a single search iteration (`num_candidates=n`, `keep_k=1`, `max_iterations=1`, `propose_mode="sample"`) whose one segment spans the whole `max_new_tokens` budget. Each sampled continuation is a full rollout, so any composed logits processor (for example RAD) steers every sample. Parameters for the scorer travel to it at inference time via `runtime_kwargs={"reward_params": {...}}`.

## Method parameters

| parameter | type | description |
| --------- | ---- | ----------- |
| `n` | `int` | Number of full-length continuations to sample and rank |
| `scorer` | `Callable` | A sequence scorer `(prompt, continuations, params) -> list[float]`; the highest-scoring sample is returned |

## Setup

If running this from a Google Colab notebook, uncomment the clone cell below. It is not necessary when running from a virtual environment where the package is already installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [2]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: reranking by keyword coverage

The scorer is any callable `(prompt, continuations, params) -> list[float]`, where `params` is whatever was passed as `reward_params` at generation time. We define a scorer that counts how many required keywords a continuation covers, then ask for a single sentence that works in all of them.

In [3]:

from transformers import AutoTokenizer, set_seed

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.best_of_n.control import BestOfN

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The scorer rewards one point per covered keyword. With sixteen required words and a 56-token budget, no single sample fits them all and most drop several, which gives reranking real headroom: the spread across samples is what selection converts into score.


In [4]:
def keyword_coverage(prompt, continuations, params):
    terms = [t.lower() for t in params.get("key_terms", [])]
    return [float(sum(term in c.lower() for term in terms)) for c in continuations]


KEY_TERMS = [
    "cat", "couch", "sun", "tea", "nap", "book", "rain", "socks",
    "lamp", "blanket", "pillow", "candle", "sweater", "toast", "radio", "slippers",
]


### Baseline: a single sample (`n=1`)

With one candidate, taking the argmax over one score is a no-op, so `n=1` is plain sampling. We fix the seed so the runs below are comparable, and print the scorer's verdict alongside the output.

In [5]:
pipeline_n1 = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=1, scorer=keyword_coverage)],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline_n1.steer()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
prompt = (
    "Write one sentence about a lazy afternoon at home that mentions all of these words: "
    "cat, couch, sun, tea, nap, book, rain, socks, lamp, blanket, pillow, candle, sweater, "
    "toast, radio, slippers."
)
chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt").to(pipeline_n1.model.device)

set_seed(42)
output = pipeline_n1.generate(
    input_ids=inputs["input_ids"],
    runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
    max_new_tokens=56,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
text_n1 = tokenizer.decode(output[0], skip_special_tokens=True)
print(text_n1)
print(f"\nkeyword score: {keyword_coverage(prompt, [text_n1], {'key_terms': KEY_TERMS})[0]:.0f}/16")



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/338 [00:00<04:00,  1.40it/s]


Loading weights:   1%|▏         | 5/338 [00:00<00:43,  7.72it/s]


Loading weights:   5%|▌         | 17/338 [00:00<00:11, 27.89it/s]


Loading weights:   7%|▋         | 25/338 [00:01<00:08, 35.01it/s]

...


Loading weights:  94%|█████████▍| 317/338 [00:07<00:00, 36.38it/s]


Loading weights:  96%|█████████▌| 325/338 [00:07<00:00, 41.04it/s]


Loading weights:  98%|█████████▊| 330/338 [00:07<00:00, 42.14it/s]


Loading weights: 100%|██████████| 338/338 [00:07<00:00, 44.43it/s]

On a rainy afternoon, my fluffy cat curled up on the comfy couch to enjoy a warm cup of tea while I napped with a blanket and pillow, reading a book under a cozy lamp, listening to the radio through my slippers, and savoring some toasted cheese for

keyword score: 12/16


A single sample is at the mercy of the sampling path it happens to take; the score records how many of the sixteen keywords it covered.


### Best of 8

Same seed, same prompt, but the driver now proposes eight full continuations, scores each with `keyword_coverage`, and returns the argmax.

In [6]:
pipeline_n8 = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=8, scorer=keyword_coverage)],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline_n8.steer()

set_seed(42)
output = pipeline_n8.generate(
    input_ids=inputs["input_ids"].to(pipeline_n8.model.device),
    runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
    max_new_tokens=56,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
text_n8 = tokenizer.decode(output[0], skip_special_tokens=True)
print(text_n8)
print(f"\nkeyword score: {keyword_coverage(prompt, [text_n8], {'key_terms': KEY_TERMS})[0]:.0f}/16")



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/338 [00:00<01:06,  5.05it/s]


Loading weights:   8%|▊         | 28/338 [00:00<00:02, 112.57it/s]


Loading weights:  19%|█▊        | 63/338 [00:00<00:01, 191.76it/s]


Loading weights:  26%|██▌       | 87/338 [00:00<00:01, 204.24it/s]

...


Loading weights:  75%|███████▌  | 255/338 [00:01<00:00, 281.25it/s]


Loading weights:  86%|████████▌ | 291/338 [00:01<00:00, 294.91it/s]


Loading weights:  95%|█████████▍| 321/338 [00:01<00:00, 290.92it/s]


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 252.00it/s]

On a rainy afternoon, the cozy cat snuggled on the soft blanket while sipping tea and napping in front of the warm lamp, surrounded by books and magazines, with the gentle sound of a radio playing softly as they savored their delicious toast and read a novel.

keyword score: 9/16


With eight candidates to choose from, the returned continuation covers more of the required keywords than the single sample did, though still nowhere near all sixteen. Nothing about the model changed; the improvement comes entirely from selection.


### What the driver does internally

One search iteration is nothing more than propose, score, keep. The cell below reproduces it directly against the same model: sample eight continuations with `num_return_sequences=8`, score them with the same scorer, and take the argmax. `BestOfN` automates exactly this loop, and generalizes it, since the pipeline's composed logits processors and stopping criteria apply to every rollout.

In [7]:
set_seed(42)
rollouts = pipeline_n8.model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=56,
    do_sample=True,
    num_return_sequences=8,
    pad_token_id=tokenizer.eos_token_id,
)
continuations = tokenizer.batch_decode(rollouts[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
scores = keyword_coverage(prompt, continuations, {"key_terms": KEY_TERMS})

for score, continuation in sorted(zip(scores, continuations), reverse=True):
    print(f"[{score:.0f}] {continuation}")

[9] On a rainy afternoon, the cozy cat snuggled on the soft blanket while sipping tea and napping in front of the warm lamp, surrounded by books and magazines, with the gentle sound of a radio playing softly as they savored their delicious toast and read a novel.
[9] On a rainy afternoon, I lazily lounged on the comfortable couch with my favorite book while sipping tea and reading, enjoying the warmth from the lamp as I napped under a fluffy blanket, watched TV with my slippers on, read another chapter, listened to my radio
[8] On a rainy day, I snuggled up with my favorite blanket and pillow on the cozy couch, sipping hot cocoa from a steaming mug while reading a good book, listening to my favorite radio station, and enjoying a slice of toast with a cup of tea as I
[8] On a rainy afternoon at home, I snuggled with my fluffy cat on the cozy couch, sipped hot tea from an old mug while napping, and listened to the gentle sound of my radio as I read a book by the warm light of a lamp.
[8]

The spread across the eight samples is the whole story of best-of-N. Every rollout misses a chunk of the sixteen words, the best cover the most, and the driver simply keeps the top row of this list.


### Scaling `n`

Each candidate is a full rollout, so best-of-N costs `n` times the decode compute of a single generation. The sweep below reads out what that compute buys on this task.

In [8]:
for n in [1, 4, 16]:
    sweep_pipeline = SteeringPipeline(
        model_name_or_path=MODEL_NAME,
        controls=[BestOfN(n=n, scorer=keyword_coverage)],
        device_map="auto",
        hf_model_kwargs={"dtype": "auto"},
    )
    sweep_pipeline.steer()
    set_seed(42)
    output = sweep_pipeline.generate(
        input_ids=inputs["input_ids"].to(sweep_pipeline.model.device),
        runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
        max_new_tokens=56,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    score = keyword_coverage(prompt, [text], {"key_terms": KEY_TERMS})[0]
    print(f"n={n:>2}  winner score: {score:.0f}/16")



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/338 [00:00<01:06,  5.05it/s]


Loading weights:   8%|▊         | 28/338 [00:00<00:02, 112.56it/s]


Loading weights:  19%|█▊        | 63/338 [00:00<00:01, 192.37it/s]


Loading weights:  26%|██▋       | 89/338 [00:00<00:01, 211.02it/s]

...


Loading weights:  75%|███████▌  | 255/338 [00:01<00:00, 292.06it/s]


Loading weights:  84%|████████▍ | 285/338 [00:01<00:00, 289.38it/s]


Loading weights:  93%|█████████▎| 315/338 [00:01<00:00, 287.24it/s]


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 252.93it/s]

n=16  winner score: 10/16


The winner's score climbs with `n`: a larger pool keeps turning up sentences that pack in more of the sixteen words, and the curve only flattens once the 56-token budget itself becomes the binding constraint. This score-versus-compute curve is the practical dial of the method.


## Example: self-consistency with `MajorityVoteScorer`

Swapping the scorer changes the method. `MajorityVoteScorer` scores each continuation by how many of the others share its extracted answer, so best-of-N with this scorer returns a continuation carrying the plurality answer over `n` sampled reasoning paths. This is self-consistency (Wang et al., 2022), obtained purely as a scorer choice.

Self-consistency needs a problem in the band where sampled reasoning paths genuinely disagree: easy enough that correct paths are common, hard enough that any single path, the greedy one included, often derails. A trivial problem gives unanimous votes and the scorer has nothing to do. We use a Level 5 problem from MATH-500 (the `HuggingFaceH4/MATH-500` subset of MATH; this is row `test/algebra/291.json`): Alice and Bob alternate coin flips with Alice starting; Alice wins on heads on her turn, Bob wins on tails on his. Alice's winning probability sums her chances on flips one, three, five, and so on, a geometric series with first term 1/2 and ratio 1/4, giving 2/3. The symmetric-looking setup tempts 1/2, and dropped or mangled terms produce other fractions, so sampled paths scatter across a few answer clusters with the correct one, ideally, in the plurality.

The scorer takes an `answer_extractor`, and the extractor defines what counts as the same vote. We anchor on a final `Answer:` line or a `\boxed{...}`, parse integers, fractions, and decimals, and canonicalize with `fractions.Fraction` so that `4/6`, `\frac{2}{3}`, and `2/3` fall into one bucket (and exact decimals like `0.5` merge with `1/2`), with a last-number fallback for responses that ignore the format.


In [9]:
import re
from fractions import Fraction

from aisteer360.algorithms.output_control.common.scorers import MajorityVoteScorer

# level 5 problem from MATH-500 (HuggingFaceH4/MATH-500, row test/algebra/291.json); the answer is 2/3
MATH_PROBLEM = (
    "Alice and Bob are playing a game. Alice starts first. On Alice's turn, she flips a coin. "
    "If she gets a heads, she wins. If not, it becomes Bob's turn. On Bob's turn, he flips a coin. "
    "If he gets a tails, he wins. If not, it becomes Alice's turn. "
    "What is the probability that Alice wins the game?"
)
MATH_ANSWER = "2/3"

ANSWER_PATTERN = re.compile(
    r"(?:Answer:|\\boxed\{)\s*\$?\\?(?:d?frac\{(-?\d+)\}\{(-?\d+)\}|(-?\d+)\s*/\s*(-?\d+)|(-?\d+(?:\.\d+)?))"
)


def extract_answer(text: str) -> str:
    matches = list(ANSWER_PATTERN.finditer(text))
    if matches:
        num1, den1, num2, den2, plain = matches[-1].groups()
        try:
            if num1 is not None:
                return str(Fraction(int(num1), int(den1)))
            if num2 is not None:
                return str(Fraction(int(num2), int(den2)))
            return str(Fraction(plain))
        except (ZeroDivisionError, ValueError):
            return ""
    numbers = re.findall(r"-?\d+(?:\.\d+)?(?:\s*/\s*-?\d+)?", text.replace(",", ""))
    if not numbers:
        return ""
    try:
        return str(Fraction(numbers[-1].replace(" ", "")))
    except (ZeroDivisionError, ValueError):
        return ""


majority_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=16, scorer=MajorityVoteScorer(answer_extractor=extract_answer))],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
majority_pipeline.steer()

math_prompt = (
    f"{MATH_PROBLEM} Work through it step by step, then end your response with "
    '"Answer: <value>", where <value> is an integer or a fraction in lowest terms.'
)
math_chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": math_prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
math_inputs = tokenizer(math_chat, return_tensors="pt").to(majority_pipeline.model.device)

set_seed(42)
output = majority_pipeline.generate(
    input_ids=math_inputs["input_ids"],
    max_new_tokens=400,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)
majority_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(majority_text)
print(f"\nextracted answer: {extract_answer(majority_text)} (target {MATH_ANSWER})")



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/338 [00:00<01:07,  5.00it/s]


Loading weights:   8%|▊         | 28/338 [00:00<00:02, 106.81it/s]


Loading weights:  19%|█▉        | 64/338 [00:00<00:01, 189.15it/s]


Loading weights:  29%|██▉       | 99/338 [00:00<00:01, 233.33it/s]

...


Loading weights:  77%|███████▋  | 261/338 [00:01<00:00, 285.54it/s]


Loading weights:  86%|████████▋ | 292/338 [00:01<00:00, 277.99it/s]


Loading weights:  97%|█████████▋| 327/338 [00:01<00:00, 297.12it/s]


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 250.97it/s]

Let's analyze this problem step by step.

1. **Probability of Alice winning on her first turn:**
   - Alice has a 50% chance (1/2) of getting heads.
   - If she gets heads, she immediately wins.
   - Therefore, the probability of Alice winning on her first turn is \( \frac{1}{2} \).

2. **Probability of Bob winning on his first turn:**
   - After Alice's turn, if she does not get heads, it becomes Bob's turn.
   - Bob also has a 50% chance (1/2) of getting tails.
   - If he gets tails, he wins.
   - Therefore, the probability of Bob winning on his first turn is \( \frac{1}{2} \).

3. **Probability of neither Alice nor Bob winning on their first turns:**
   - For this to happen, both must fail on their respective turns.
   - The probability of Alice failing on her first turn is \( \frac{1}{2} \).
   - The probability of Bob failing on his first turn is \( \frac{1}{2} \).
   - Thus, the combined probability for this scenario is \( \frac{1}{2} \times \frac{1}{2} = \frac{1}{4} \).

4. **Co

### Comparison: a single greedy answer

The self-consistency claim is that the plurality over sampled reasoning paths beats the single path greedy decoding commits to. For the comparison we decode the same prompt greedily, without the driver.

In [10]:
greedy_ids = majority_pipeline.model.generate(
    input_ids=math_inputs["input_ids"],
    attention_mask=math_inputs["attention_mask"],
    max_new_tokens=400,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
greedy_text = tokenizer.decode(greedy_ids[0][math_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(greedy_text)
print(f"\nextracted answer: {extract_answer(greedy_text)} (target {MATH_ANSWER})")


To determine the probability that Alice wins the game, we need to consider all possible outcomes of the coin flips and how they affect the game.

1. **First Flip (Alice's Turn):**
   - Probability of getting heads (Alice wins) = 1/2
   - Probability of getting tails (it moves to Bob) = 1/2

2. **Second Flip (Bob's Turn):**
   - If Alice got tails on her first flip:
     - Probability of getting tails (Bob wins) = 1/2
     - Probability of getting heads (it returns to Alice) = 1/2

Now let's break down the scenarios:

- **Scenario 1:** Alice wins on her first flip.
  - Probability = 1/2

- **Scenario 2:** Alice loses on her first flip but wins on her second flip.
  - Probability of losing on Alice's first flip = 1/2
  - Probability of winning on Bob's first flip = 1/2
  - Probability of winning on Alice's second flip = 1/2
  - Combined probability for this scenario = (1/2) * (1/2) * (1/2) = 1/8

The total probability that Alice wins the game is the sum of the probabilities from both sce

The correct answer is 2/3. A single reasoning path has to survive every step of the setup, and greedy decoding commits to exactly one such path with no recourse; whether it happens to be a sound one is a property of the model and prompt, not something the decoding protects. The plurality over sixteen sampled paths is the robust object: wrong paths scatter across minority clusters while correct paths agree, so the majority scorer keeps returning 2/3 even when a large fraction of individual samples is wrong (Wang et al.'s result). As problems harden past what a single path reliably solves, that robustness gap is the whole value of the method.

### The vote, made visible

The driver's argmax is over agreement counts, so the informative object is the histogram of extracted answers across the pool. The cell below redoes the proposal step directly, sampling sixteen continuations, and tabulates the votes the scorer counted. The plurality cluster is what `BestOfN` returned above; the minority clusters are the derailments that any single sample, greedy included, risks committing to (the symmetric 1/2 reading is a common one). Note also how the extractor's canonicalization shapes the buckets: a sloppier extractor that kept `4/6` and `2/3` apart would split the correct vote.


In [11]:
from collections import Counter

set_seed(42)
rollouts = majority_pipeline.model.generate(
    input_ids=math_inputs["input_ids"],
    attention_mask=math_inputs["attention_mask"],
    max_new_tokens=400,
    do_sample=True,
    temperature=0.8,
    num_return_sequences=16,
    pad_token_id=tokenizer.eos_token_id,
)
continuations = tokenizer.batch_decode(rollouts[:, math_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
votes = Counter(extract_answer(c) for c in continuations)

for answer, count in votes.most_common():
    label = answer if answer else "-"
    marker = "  <- correct" if answer == MATH_ANSWER else ""
    print(f"{label:>8}: {'#' * count} ({count}/16){marker}")


       2: ######## (8/16)
       4: #### (4/16)
       1: # (1/16)
     2/3: # (1/16)  <- correct
     1/2: # (1/16)
       3: # (1/16)


### Takeaway

Best-of-N is the first thing to try when you can score what you want: it needs no training, composes with everything, and costs a transparent `n` full decodes per output. The scorer is the method, as this notebook shows twice with the same driver (keyword reranking, then self-consistency via `MajorityVoteScorer`; the shipped scorers live in `aisteer360.algorithms.output_control.common.scorers`).

Because every candidate is a full rollout through the composed stacks, a step-level control steers all `n` samples; running RAD under `BestOfN` reranks already-detoxified candidates ([rad.ipynb](rad.ipynb)). For iterative segment-level search with the same scorer contract, see DeAL ([deal.ipynb](deal.ipynb)). See the [output control](https://ibm.github.io/AISteer360/concepts/controls/#output-control) section of the docs for the full family.
